# Assignment 2 – Exercise 2: Deep Ensemble OOD Detection


In [1]:
import json, torch, timm, numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')


Device: cpu


In [2]:
ds_raw = load_dataset('blanchon/UC_Merced')['train']
label_names = ds_raw.features['label'].names

splits_path = Path('../Assignment1/splits.json')
if not splits_path.exists():
    splits_path = Path('splits.json')
with open(splits_path) as f:
    splits = json.load(f)

class UCMercedDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        item = ds_raw[self.indices[idx]]
        img = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

class FIDS30Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = list(Path(root_dir).rglob('*.jpg')) + list(Path(root_dir).rglob('*.png'))
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, -1

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds   = UCMercedDataset(splits['train'], train_tf)
test_ds    = UCMercedDataset(splits['test'],  val_tf)
fids30_ds  = FIDS30Dataset('PrepData/FIDS30', val_tf)
train_loader  = DataLoader(train_ds,  batch_size=32, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_ds,   batch_size=32, shuffle=False, num_workers=0)
fids30_loader = DataLoader(fids30_ds, batch_size=32, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}, Test: {len(test_ds)}, FIDS30: {len(fids30_ds)}')


Train: 1470, Test: 315, FIDS30: 971


## Train Deep Ensemble (10 × EfficientNet-B0, 5 epochs each)


In [3]:
Path('models').mkdir(exist_ok=True)
N_MODELS = 10
EPOCHS = 5

def train_single_model(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=21).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        print(f'  [seed={seed}] Epoch {epoch}/{EPOCHS} done', flush=True)
    return model

ensemble_models = []
for i in range(N_MODELS):
    ckpt = f'models/ensemble_model_{i}.pth'
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=21).to(device)
    if Path(ckpt).exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f'Loaded model {i} from checkpoint')
    else:
        print(f'Training model {i} (seed={i})...')
        model = train_single_model(i)
        torch.save(model.state_dict(), ckpt)
        print(f'Saved model {i} -> {ckpt}')
    ensemble_models.append(model)
print(f'Ensemble ready: {len(ensemble_models)} models')


Training model 0 (seed=0)...


  [seed=0] Epoch 1/5 done


## Inference: Collect Softmax Outputs


In [ ]:
def get_softmax_outputs(models, loader, device):
    all_model_outputs = []
    for model in models:
        model.eval()
        outputs = []
        with torch.no_grad():
            for x, _ in loader:
                probs = F.softmax(model(x.to(device)), dim=1)
                outputs.append(probs.cpu().numpy())
        all_model_outputs.append(np.concatenate(outputs, axis=0))
    return np.stack(all_model_outputs, axis=0)

print('Running inference on UC Merced test set...')
uc_softmax   = get_softmax_outputs(ensemble_models, test_loader,   device)
print('Running inference on FIDS30...')
fids_softmax = get_softmax_outputs(ensemble_models, fids30_loader, device)
print(f'UC Merced softmax shape:  {uc_softmax.shape}')
print(f'FIDS30 softmax shape:     {fids_softmax.shape}')


## OOD Score: Variance Across Models + Threshold


In [ ]:
uc_variance   = uc_softmax.var(axis=0).mean(axis=1)
fids_variance = fids_softmax.var(axis=0).mean(axis=1)

threshold = np.percentile(uc_variance, 95)
false_alarm_rate = (uc_variance   > threshold).mean()
detection_rate   = (fids_variance > threshold).mean()

print(f'Threshold (95th pct of ID variance): {threshold:.6f}')
print(f'False Alarm Rate  (UC Merced > threshold): {false_alarm_rate:.4f}  ({false_alarm_rate*100:.1f}%)')
print(f'OOD Detection Rate (FIDS30 > threshold):  {detection_rate:.4f}  ({detection_rate*100:.1f}%)')


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(uc_variance,   bins=60, alpha=0.6, color='steelblue', label='UC Merced (In-Dist)', density=True)
plt.hist(fids_variance, bins=60, alpha=0.6, color='tomato',    label='FIDS30 (OOD)',        density=True)
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'95th pct threshold = {threshold:.5f}')
plt.xlabel('OOD Score (Ensemble Variance)')
plt.ylabel('Density')
plt.title('Deep Ensemble OOD Detection – Score Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('ensemble_ood_histogram.png', dpi=100)
plt.show()
print(f'Detection Rate: {detection_rate*100:.1f}%  |  False Alarm Rate: {false_alarm_rate*100:.1f}%')
